In [79]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

In [82]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_fix.xlsx")

In [83]:
claude_label.shape

(5000, 10)

In [84]:
claude_label

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer,artistic_value_comment,creativity_answer,creativity_comment
0,230,Buste de femme (Dora Maar),Pablo,Picasso,1939,Spanish,High,Picasso's depiction uses the interaction of pa...,Yes,Dora aesthetically stimulated Picasso in a way...
1,896,Mousquetaire buste,Pablo,Picasso,1968,Spanish,High,The Mousquetaire. Buste of 1968 belongs to a m...,Yes,"These portraits represented a late flowering, ..."
2,1241,Le transformateur,Pablo,Picasso,1953,Spanish,High,The Transformateur paintings serve as an index...,Yes,These paintings represent genuine creative inn...
3,1596,Le peintre et son modèle,Pablo,Picasso,1964,Spanish,High,"In Picasso's later years, the theme of painter...",Yes,The work demonstrates residual Cubism through ...
4,1733,Portrait de Sylvette,Pablo,Picasso,1954,Spanish,High,The Portrait de Sylvette is recognized by muse...,Yes,The Sylvette series represents the most concen...
...,...,...,...,...,...,...,...,...,...,...
4995,424928667,Au théatre,Pablo,Picasso,1966,Spanish,Unable to Determine,The available sources do not provide sufficien...,Unable to Determine,To properly assess creativity according to you...
4996,424928669,Couverture Mourlot III,Pablo,Picasso,1956,Spanish,Limited Evidence in Sources,The work was created as the cover of the refer...,Insufficient Critical Evidence,The search results do not provide expert analy...
4997,424931542,Face of May,Pablo,Picasso,1946,Spanish,High,"After World War II, Picasso began working inte...",Yes,"Picasso made lithographs since the 1920s, but ..."
4998,424934832,Fleur bleue,Pablo,Picasso,1964,Spanish,"authoritative commentary on Picasso's ""Fleur b...","To provide you with accurate, authoritative an...","for authoritative commentary on Picasso's ""Fle...","To provide you with accurate, authoritative an..."


In [85]:
sum(claude_label['artwork id'].duplicated())

0

In [86]:
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_fix.xlsx")

In [87]:
gemini_label.shape

(5000, 10)

In [88]:
sum(gemini_label['artwork id'].duplicated())

0

In [89]:
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_final.xlsx")

In [90]:
openai_label.shape

(5000, 10)

In [91]:
sum(openai_label['artwork id'].duplicated())

0

In [92]:
full_df = claude_label.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_openai"))

In [93]:
full_df.shape

(5000, 14)

In [94]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_openai,artistic_value_comment_openai,creativity_answer_openai,creativity_comment_openai
0,230,Buste de femme (Dora Maar),Pablo,Picasso,1939,Spanish,High,Picasso's depiction uses the interaction of pa...,Yes,Dora aesthetically stimulated Picasso in a way...,High,"""Buste de femme (Dora Maar)"" is a significant ...",Yes,"Picasso's ""Buste de femme (Dora Maar)"" showcas..."
1,896,Mousquetaire buste,Pablo,Picasso,1968,Spanish,High,The Mousquetaire. Buste of 1968 belongs to a m...,Yes,"These portraits represented a late flowering, ...",High,Pablo Picasso's 'Mousquetaire buste' (1968) ex...,Yes,'Mousquetaire buste' demonstrates Picasso's in...
2,1241,Le transformateur,Pablo,Picasso,1953,Spanish,High,The Transformateur paintings serve as an index...,Yes,These paintings represent genuine creative inn...,High,"""Le transformateur"" is a significant work with...",Yes,"""Le transformateur"" exemplifies Picasso's crea..."
3,1596,Le peintre et son modèle,Pablo,Picasso,1964,Spanish,High,"In Picasso's later years, the theme of painter...",Yes,The work demonstrates residual Cubism through ...,High,"""Le Peintre et Son Modèle"" is a significant wo...",Yes,"In ""Le Peintre et Son Modèle,"" Picasso innovat..."
4,1733,Portrait de Sylvette,Pablo,Picasso,1954,Spanish,High,The Portrait de Sylvette is recognized by muse...,Yes,The Sylvette series represents the most concen...,High,"Pablo Picasso's ""Portrait of Sylvette"" (1954) ...",Yes,"""Portrait of Sylvette"" exemplifies Picasso's c..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,424928667,Au théatre,Pablo,Picasso,1966,Spanish,Unable to Determine,The available sources do not provide sufficien...,Unable to Determine,To properly assess creativity according to you...,High,"Pablo Picasso's 1966 work, ""Au théâtre,"" exemp...",Yes,"In ""Au théâtre,"" Picasso's innovative approach..."
4996,424928669,Couverture Mourlot III,Pablo,Picasso,1956,Spanish,Limited Evidence in Sources,The work was created as the cover of the refer...,Insufficient Critical Evidence,The search results do not provide expert analy...,High,"""Couverture Mourlot III"" is a lithograph creat...",Yes,"Picasso's ""Couverture Mourlot III"" demonstrate..."
4997,424931542,Face of May,Pablo,Picasso,1946,Spanish,High,"After World War II, Picasso began working inte...",Yes,"Picasso made lithographs since the 1920s, but ...",High,"Pablo Picasso's 1946 painting, ""Face of May,"" ...",Yes,"""Face of May"" exemplifies Picasso's creativity..."
4998,424934832,Fleur bleue,Pablo,Picasso,1964,Spanish,"authoritative commentary on Picasso's ""Fleur b...","To provide you with accurate, authoritative an...","for authoritative commentary on Picasso's ""Fle...","To provide you with accurate, authoritative an...",High,"Pablo Picasso's 1964 work, 'Fleur bleue,' exem...",Yes,'Fleur bleue' demonstrates Picasso's continuou...


In [95]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [96]:
full_df.shape

(5000, 18)

# Embedding Convert

In [97]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

In [98]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = text_model.to(device)

In [99]:
def convert_one_row(model,i,texts,device):
    try:
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        text_embeds = outputs.pooler_output
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
        text_embeds=text_embeds.cpu().numpy()
    except Exception as e:
        print(f"Error processing {i}: {e}")
        text_embeds = np.zeros([2,512])
    return i, text_embeds

In [104]:
dataset = "gemini"
if dataset =="claude":
    df = claude_label.copy()
elif dataset =="gemini":
    df = gemini_label.copy()
elif dataset =="openai":
    df = openai_label.copy()

In [105]:
number_size=df.shape[0]
#number_size=10
# range_start = 30000
range_start = 0
range_end = min(range_start+number_size,df.shape[0])
N = min(number_size, df.shape[0]-range_start)

In [106]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, 
                  [df.iloc[i].artistic_value_comment,df.iloc[i].creativity_comment],
                  device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, text_embeds = fut.result()
        embeddings[i-range_start] =text_embeds
        if i % 1000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-04 16:02:51: Start
2025-12-04 16:02:53: 0
Error processing 108: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 246: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 282: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 477: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 752: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples).
Error processing 766: text input must be of type `str` 

In [107]:
np.save(f"clip_embeddings_{dataset}_final.npy", embeddings)

# Comment Check Consistency Rough

In [152]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini.strip()):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini.strip()):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [164]:
np.sum(consistent_artist)

np.int64(78)

In [163]:
np.sum(consistent_creative)

np.int64(73)

In [153]:
np.sum(consistent_overall)

np.int64(73)

In [154]:
check=full_df.copy()
check["creative_consist"]=consistent_creative
check["artistic_consist"]=consistent_artist
check["overall_consist"]=consistent_overall

# Comment Check Consistency Hard

In [155]:
claude_embed = np.load(f"clip_embeddings_claude.npy",allow_pickle=True)

In [156]:
gemini_embed = np.load(f"clip_embeddings_gemini.npy",allow_pickle=True)

In [157]:
openai_embed = np.load(f"clip_embeddings_openai.npy",allow_pickle=True)

In [158]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > 0.70).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > 0.70).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")

2025-11-28 08:04:09: Start
2025-11-28 08:04:09: Currently at 0


In [159]:
np.sum(embed_consistent_overall)

np.int64(20)

In [165]:
np.sum(embed_consistent_creative)

np.int64(31)

In [160]:
check["embed_artistic_consist"]=embed_consistent_artistic
check["embed_creative_consist"]=embed_consistent_creative
check["embed_overall_consist"]=embed_consistent_overall